In [6]:
# We import dataclass to represent 1 gene
from dataclasses import dataclass
import random
import pandas as pd
import numpy as np
from deap import base, creator, tools, algorithms
from deap.tools import cxOnePoint, cxTwoPoints
import csv

In [7]:
# A gene contains 2 attributes : the food item's name and the number of servings of this item
@dataclass
class Gene:
    food_item_name: str
    serving: int

In [8]:
# We make a list representing all the meals we have in 1 chromosome 
Meals =["Breakfast", "Snack 1", "Lunch", "Snack 2", "Dinner"]
# We make a list representing all the food groups we have in 1 meal
Food_Groups =["Vegetables", "Fruits", "Grains", "Protein", "Dairy", "Fats and Oils"]

In [9]:
import csv

def load_food_data(csv_path):
    food_data = {}
    meal_flags = {}

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            group = row['Category']
            food = row['Food Item']

            if group not in food_data:
                food_data[group] = {}

            # Safely convert numeric columns, use 0 if empty
            def safe_float(x):
                try:
                    return float(x)
                except (ValueError, TypeError):
                    return 0.0

            food_data[group][food] = {
                'p': safe_float(row['Protein (g)']),
                'f': safe_float(row['Fats (g)']),
                'c': safe_float(row['Carbohydrates (g)']),
                'cal': safe_float(row['Calories (kcal)'])
            }

            # Safely convert meal flags (0 if empty)
            def safe_int(x):
                try:
                    return int(x)
                except (ValueError, TypeError):
                    return 0

            meal_flags[food] = {
                'Breakfast': safe_int(row['Breakfast']),
                'Snack 1': safe_int(row['Snack 1']),
                'Lunch': safe_int(row['Lunch']),
                'Snack 2': safe_int(row['Snack 2']),
                'Dinner': safe_int(row['Dinner'])
            }

    return food_data, meal_flags

FOOD_DATA, MEAL_FLAGS = load_food_data("Processed_Bahrain_Food_Dataset.csv")

In [10]:
# I created a dictionary that contains all the saudi guideline's servings
SERVING_GUIDELINES_BY_KCAL = {
    1200: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 0},
        "Snack 2":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1300: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1400: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1500: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1600: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1700: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1800: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    },

    1900: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    },

    2000: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 2, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 2, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    2200: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1":    {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch":     {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2":    {"Grains": 2, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner":    {"Grains": 2, "Dairy": 0, "Protein": 2, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    }
}

In [11]:
def print_meals(ind, title=""):
    print(f"\n{title}")
    print("-" * 60)
    for m in range(5):
        meal_name = Meals[m]
        genes = ind[m*6:(m+1)*6]
        foods = [g.food_item_name for g in genes]
        print(f"{meal_name:<10}: ", end="")
        print(foods)

In [17]:
# DEAP Configuration
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

def create_individual():
    # Initializes 30 genes following 1600 kcal
    genes = []
    for meal in Meals:
        for group in Food_Groups:
            # Get the fixed serving count for this meal/group
            serving_count = SERVING_GUIDELINES_BY_KCAL[TARGET_KCAL][meal][group]

           # Filter foods allowed for this meal
            allowed_foods = [
                food for food in FOOD_DATA[group].keys() if MEAL_FLAGS[food][meal] == 1
            ]
            if not allowed_foods:
                allowed_foods = list(FOOD_DATA[group].keys())  # fallback

            # Randomly select food from allowed foods
            food_name = random.choice(allowed_foods)

            # Create gene with guideline serving count
            genes.append(Gene(food_item_name=food_name, serving=serving_count))

    return creator.Individual(genes)

toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def mutate_debug(individual):
    print_meals(individual, "BEFORE MUTATION")
    ind, = mutate_meal_plan(individual)
    print_meals(ind, "AFTER MUTATION")
    return ind,

# Mutation
def mutate_meal_plan(individual):
    mutations_done = 0
    all_indices = list(range(len(individual))) # List of all possible indices in the chromosome
    random.shuffle(all_indices) # Shuffle to pick randomly without repeating the same one

    for i in all_indices:
        if mutations_done >= 2:
            break  # Stop once we have mutated 2 genes

        # Check if the serving size is not zero
        if individual[i].serving > 0:
            # Find which group it belongs to (Vegetables, Grains, ...)
            group_name = Food_Groups[i % 6]
            # Pick a new food name from food group dictionary, but the serving remains the same
            individual[i].food_item_name = random.choice(list(FOOD_DATA[group_name].keys()))
            mutations_done += 1
    return individual,

def cx_meal_level_debug(parent1, parent2):
    GROUPS = 6
    MEALS = 5

    # Choose 2 meal cut points
    m1, m2 = sorted(random.sample(range(1, MEALS), 2))
    cut1, cut2 = m1 * GROUPS, m2 * GROUPS

    print("\n========== CROSSOVER DEBUG ==========")
    print(f"Meal cut points: {m1} → {m2}")

    print_meals(parent1, "Parent 1 BEFORE")
    print_meals(parent2, "Parent 2 BEFORE")

    # Swap full meals
    parent1[cut1:cut2], parent2[cut1:cut2] = parent2[cut1:cut2], parent1[cut1:cut2]

    print_meals(parent1, "Parent 1 AFTER")
    print_meals(parent2, "Parent 2 AFTER")
    print("====================================")

    return parent1, parent2


def evaluate_meal_plan(individual):
    total_p = 0.0 # Protein grams
    total_f = 0.0 # Fat grams
    total_c = 0.0 # Carbohydrates grams
    total_cal = 0.0 # Total Calories

    # First: Sum up all grams and calories based on food item and serving count
    for gene in individual:
        group_name = None
        # Find which group this food belongs to
        for group, foods in FOOD_DATA.items():
            if gene.food_item_name in foods:
                group_name = group
                break

        if group_name:
            stats = FOOD_DATA[group_name][gene.food_item_name]
            total_p += stats['p'] * gene.serving
            total_f += stats['f'] * gene.serving
            total_c += stats['c'] * gene.serving
            total_cal += stats['cal'] * gene.serving

    # Second : Calculate actual macro percentages (p_c, p_p, p_f)
    g_total = total_p + total_f + total_c
    if g_total == 0: return 0, # Avoid division by zero

    p_p = total_p / g_total
    p_f = total_f / g_total
    p_c = total_c / g_total

    # Third :Target Ratios (The AMDR values)
    r_p, r_f, r_c = 0.20, 0.25, 0.55
    # Fourth: Calculate J_macro (Squared Error)
    j_macro = (p_c - r_c)**2 + (p_p - r_p)**2 + (p_f - r_f)**2

    # Fifth: Calculate J_cal
    r = (total_cal - TARGET_KCAL) / TARGET_KCAL
    j_cal = min(1, r**2)

    # Sixth: apply the variety constraint
    N = 30 # total number of food items in the chromosome ( N is taken from the report)
    # count food appearance
    food_appearance={}
    for gene in individual:
        food_appearance[gene.food_item_name]= food_appearance.get(gene.food_item_name, 0) + 1

    # Calculate the penalty for variety
    j_var = 0
    for food_name,count in food_appearance.items():
        # we penalize if the food item appears more than once
        if count > 1:
            j_var += max(0,count-1)**2
    # Normalize the penalty
    j_var = j_var / ((N-1)**2)

   # Seventh: calculate the score
    w_macro, w_var, w_cal = 0.33, 0.33, 0.33 #change to 0.25 when adding preference
    score = w_macro * j_macro + w_var * j_var + w_cal * j_cal
    # store the penalties to print later
    individual.j_var= j_var
    individual.j_macro = j_macro
    individual.j_cal = j_cal

    print("\n--- FITNESS DEBUG ---")
    print(f"Calories: {total_cal:.1f}")
    print(f"Macros %: C={p_c:.2f}, P={p_p:.2f}, F={p_f:.2f}")
    print(f"J_macro={j_macro:.4f}, J_var={j_var:.4f}, J_cal={j_cal:.4f}")

    # Lastly: We want to minimize score, so we return a fitness that increases as score decreases
    return 1 / (1 + score),


toolbox.register("evaluate", evaluate_meal_plan)
import random

def cx_one_point_debug(ind1, ind2):
    print("\n========== CROSSOVER DEBUG ==========")

    # Print parents BEFORE crossover
    print_meals(ind1, "Parent 1 BEFORE")
    print_meals(ind2, "Parent 2 BEFORE")

    size = min(len(ind1), len(ind2))

    if size < 2:
        return ind1, ind2  # nothing to crossover

    # This is EXACTLY what DEAP does internally
    cxpoint = random.randint(1, size - 1)

    print(f"Crossover point index: {cxpoint}")

    # Perform one-point crossover
    ind1[cxpoint:], ind2[cxpoint:] = ind2[cxpoint:], ind1[cxpoint:]

    # Print parents AFTER crossover
    print_meals(ind1, "Parent 1 AFTER")
    print_meals(ind2, "Parent 2 AFTER")

    print("====================================")

    return ind1, ind2

toolbox.register("mate", cx_one_point_debug)

toolbox.register("select", tools.selTournament, tournsize=3) # Tournament selection
toolbox.register("mutate", mutate_debug)

# Execution Loop
def run_simulation(target_kcal):
    global TARGET_KCAL
    TARGET_KCAL = target_kcal   # tells create_individual which guideline to use

    pop = toolbox.population(n=10) # Population

    print_meals(pop[0], "INITIAL INDIVIDUAL CHECK")

    hof = tools.HallOfFame(1) # Keeps the absolute best found across all generations

    mu = len(pop) # Number of individuals to select for the next generation
    lambd = 10 # Number of children to produce in each generation

    # Statistics object
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Run Elitism-Based GA
    algorithms.eaMuPlusLambda(pop, toolbox,
                               mu=mu,
                               lambda_=lambd,
                               cxpb=0.7,   # Crossover probability
                               mutpb=0.2,  # Mutation probability
                               ngen=2,    # Number of generations
                               stats=stats,
                               halloffame=hof,
                               verbose=True)

    return hof[0]

In [13]:
'''
GMAX = 100
NO_IMPROVEMENT_LIMIT = 5
EPS = 1e-6

def run_simulation(target_kcal):
    global TARGET_KCAL
    TARGET_KCAL = target_kcal

    # Initialization
    pop = toolbox.population(n=50)
    hof = tools.HallOfFame(1)

    best_fitness_so_far = None
    stagnant_generations = 0

    # Statistics object
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    logbook = tools.Logbook()
    logbook.header = ["gen", "avg", "min", "max"]

    for entry in logbook:
     print(entry)

    # Evolution loop
    for _ in range(GMAX):

        # Run ONE generation
        algorithms.eaMuPlusLambda(
            pop,
            toolbox,
            mu=len(pop),
            lambda_=len(pop),
            cxpb=0.7,
            mutpb=0.2,
            ngen=1,
            halloffame=hof,
            verbose=False
        )

        current_best = hof[0].fitness.values[0]

        # Improvement check
        if best_fitness_so_far is None or current_best > best_fitness_so_far + EPS:
            best_fitness_so_far = current_best
            stagnant_generations = 0
        else:
            stagnant_generations += 1

        # Early stopping condition
        if stagnant_generations >= NO_IMPROVEMENT_LIMIT:
            break

    return hof[0]
'''

'\nGMAX = 100\nNO_IMPROVEMENT_LIMIT = 5\nEPS = 1e-6\n\ndef run_simulation(target_kcal):\n    global TARGET_KCAL\n    TARGET_KCAL = target_kcal\n\n    # Initialization\n    pop = toolbox.population(n=50)\n    hof = tools.HallOfFame(1)\n\n    best_fitness_so_far = None\n    stagnant_generations = 0\n\n    # Statistics object\n    stats = tools.Statistics(lambda ind: ind.fitness.values[0])\n    stats.register("avg", np.mean)\n    stats.register("min", np.min)\n    stats.register("max", np.max)\n\n    logbook = tools.Logbook()\n    logbook.header = ["gen", "avg", "min", "max"]\n\n    for entry in logbook:\n     print(entry)\n\n    # Evolution loop\n    for _ in range(GMAX):\n\n        # Run ONE generation\n        algorithms.eaMuPlusLambda(\n            pop,\n            toolbox,\n            mu=len(pop),\n            lambda_=len(pop),\n            cxpb=0.7,\n            mutpb=0.2,\n            ngen=1,\n            halloffame=hof,\n            verbose=False\n        )\n\n        current_be

In [18]:
# Execute the Simulation
best_plan = run_simulation(1600)

# Calculate Final Nutritional Totals
total_p, total_f, total_c, total_cal = 0.0, 0.0, 0.0, 0.0

print(f"\n{'MEAL':<12} | {'GROUP':<14} | {'FOOD ITEM':<25} | {'SERV.':<6} | {'P (g)':<6} | {'F (g)':<6} | {'C (g)':<6} | {'CALS':<6}")
print("-" * 75)

for i, gene in enumerate(best_plan):
    meal = Meals[i // 6]
    group = Food_Groups[i % 6]

    # Get food stats from dataset
    stats = FOOD_DATA[group][gene.food_item_name]

    # Calculate totals for this gene
    p = stats['p'] * gene.serving
    f = stats['f'] * gene.serving
    c = stats['c'] * gene.serving
    cal = stats['cal'] * gene.serving

    # Update running totals
    total_p += p
    total_f += f
    total_c += c
    total_cal += cal

    # Print row (only if servings > 0 for clarity)
    if gene.serving > 0:
        print(f"{meal:<12} | {group:<14} | {gene.food_item_name:<25} | {gene.serving:<6} | {p:<6.1f} | {f:<6.1f} | {c:<6.1f} | {cal:<6.1f}")

# Calculate Final Percentages
g_total = total_p + total_f + total_c
p_p = (total_p / g_total) * 100
p_f = (total_f / g_total) * 100
p_c = (total_c / g_total) * 100
variety= (1 - best_plan.j_var) * 100
calories = (TARGET_KCAL / total_cal) * 100

# Print Summary Report
print("\n" + "="*40)
print("       DAILY NUTRITION SUMMARY")
print("="*40)
print(f"Total Energy:   {total_cal:.1f} kcal")
print(f"Total Weight:   {g_total:.1f} g (Macro Grams)")
print("-" * 40)
print(f"Nutrient     | Actual % | Target %")
print(f"Carbs (c)    | {p_c:>7.1f}% | 55.0%")
print(f"Protein (p)  | {p_p:>7.1f}% | 20.0%")
print(f"Fats (f)     | {p_f:>7.1f}% | 25.0%")
print(f"Meal variety | {variety:>7.1f}% | 100%")
print(f"Meal calorie | {calories:>7.1f}% | 100%")
print("="*40)


INITIAL INDIVIDUAL CHECK
------------------------------------------------------------
Breakfast : ['Green Beans', 'Apple', 'Wheat Grains/Berries', 'Turkey Egg', 'Low-Fat Yogurt', 'Margarine (Unsalted)']
Snack 1   : ['Red Radish', 'Cantaloupe', 'Cooked Oatmeal', 'Egg White', 'Evaporated Milk', 'Corn Oil']
Lunch     : ['Vegetable Salad', 'Avocado', 'Cooked Noodles', 'Grilled Meed Fish', 'Fat-Free Yogurt', 'Margarine (Salted)']
Snack 2   : ['Eggplant (Aubergine)', 'Dried Pear', 'Corn Bran', 'Goose Egg', 'Low-Fat Mozzarella Cheese', 'Margarine (Salted)']
Dinner    : ['Lupin Beans', 'Blueberries', 'Cooked Noodles', 'Broasted Chicken Breast', 'Fat-Free Fruit Yogurt', 'Palm Oil']

--- FITNESS DEBUG ---
Calories: 3261.0
Macros %: C=0.66, P=0.25, F=0.09
J_macro=0.0405, J_var=0.0024, J_cal=1.0000

--- FITNESS DEBUG ---
Calories: 3261.1
Macros %: C=0.67, P=0.21, F=0.11
J_macro=0.0350, J_var=0.0071, J_cal=1.0000

--- FITNESS DEBUG ---
Calories: 3096.0
Macros %: C=0.47, P=0.34, F=0.19
J_macro=0.03